In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input

In [2]:
# Dataset Path
BASE_DIR = "plantvillage dataset/plantvillage dataset/color"

IMG_SIZE = 224
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    BASE_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

val_generator = train_datagen.flow_from_directory(
    BASE_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=32,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

Found 43456 images belonging to 38 classes.
Found 10849 images belonging to 38 classes.


In [3]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D

In [4]:
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False  # freeze base model

In [8]:
from tensorflow.keras.layers import Input
from tensorflow.keras.models import Model

# Input layer
inputs = Input(shape=(224, 224, 3))

# Pass input through EfficientNet base
x = base_model(inputs, training=False)

# Global Average Pooling
x = GlobalAveragePooling2D()(x)

# Dense layer
x = Dense(128, activation='relu')(x)

# Dropout
x = Dropout(0.5)(x)

# Output layer (38 classes)
outputs = Dense(38, activation='softmax')(x)

# Final Model
model = Model(inputs, outputs)

model.summary()


Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 efficientnetb0 (Functional  (None, 7, 7, 1280)        4049571   
 )                                                               
                                                                 
 global_average_pooling2d (  (None, 1280)              0         
 GlobalAveragePooling2D)                                         
                                                                 
 dense (Dense)               (None, 128)               163968    
                                                                 
 dropout (Dropout)           (None, 128)               0         
                                                                 
 dense_1 (Dense)             (None, 38)                4902  

In [9]:
from tensorflow.keras.optimizers import Adam

model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


In [10]:
print("Model compiled successfully ✅")


Model compiled successfully ✅


In [11]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20
)

Epoch 1/20


1358/1358 [==============================] - 2823s 2s/step - loss: 1.5942 - accuracy: 0.6035 - val_loss: 0.6280 - val_accuracy: 0.8711
Epoch 2/20
1358/1358 [==============================] - 12312s 9s/step - loss: 0.6703 - accuracy: 0.8192 - val_loss: 0.3502 - val_accuracy: 0.9217
Epoch 3/20
1358/1358 [==============================] - 1588s 1s/step - loss: 0.4671 - accuracy: 0.8697 - val_loss: 0.2530 - val_accuracy: 0.9371
Epoch 4/20
1358/1358 [==============================] - 2571s 2s/step - loss: 0.3648 - accuracy: 0.8961 - val_loss: 0.2018 - val_accuracy: 0.9476
Epoch 5/20
1358/1358 [==============================] - 2415s 2s/step - loss: 0.3010 - accuracy: 0.9145 - val_loss: 0.1695 - val_accuracy: 0.9563
Epoch 6/20
1358/1358 [==============================] - 2570s 2s/step - loss: 0.2601 - accuracy: 0.9248 - val_loss: 0.1483 - val_accuracy: 0.9605
Epoch 7/20
1358/1358 [==============================] - 1609s 1s/step - loss: 0.2284 - accuracy: 0.9343 - val_loss: 0.130

In [12]:
model.save("efficientnet_crop_disease_model.keras")


In [15]:
print(os.listdir("plantvillage dataset"))

['plantvillage dataset']


In [16]:
print(os.listdir("plantvillage dataset/plantvillage dataset/color"))

['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry_(including_sour)___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___healthy', 'Corn_(maize)___Northern_Leaf_Blight', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___healthy', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___healthy', 'Potato___Late_blight', 'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew', 'Strawberry___healthy', 'Strawberry___Leaf_scorch', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___healthy', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot', 'Tomato___Spider_mites Two-spotted_spider_mite',

In [18]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
val_generator.reset()
preds = model.predict(val_generator)
y_pred = np.argmax(preds, axis=1)
y_true = val_generator.classes
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, 
target_names=list(val_generator.class_indices.keys())))
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_true, y_pred))

340/340 [==============================] - 421s 1s/step

Classification Report:

                                                    precision    recall  f1-score   support

                                Apple___Apple_scab       0.97      0.96      0.96       126
                                 Apple___Black_rot       1.00      0.98      0.99       124
                          Apple___Cedar_apple_rust       1.00      0.98      0.99        55
                                   Apple___healthy       0.98      0.99      0.98       329
                               Blueberry___healthy       1.00      1.00      1.00       300
          Cherry_(including_sour)___Powdery_mildew       1.00      1.00      1.00       210
                 Cherry_(including_sour)___healthy       1.00      0.99      0.99       170
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot       0.84      0.84      0.84       102
                       Corn_(maize)___Common_rust_       1.00      1.00      1.00       23